# CNN Binary Classification Tutorial
Bu notebook adım adım basit bir CNN ile binary sınıflandırma gösterir: veri yükleme, ön işleme, model tasarımı, eğitim, kaydetme/yükleme, değerlendirme (accuracy, precision, recall, confusion matrix) ve Grad-CAM ısı haritası.
Kullanım: dataset'in `dataset_split/binary/train`, `.../val`, `.../test` şeklinde hazır olduğunu varsayar.

In [ ]:
# 1) Importlar ve ayarlar
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
from sklearn.metrics import classification_report, confusion_matrix
import itertools

# Reproducibility
SEED = 1337
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Paths (çalışma dizinine göre ayarlayın)
BASE_DIR = Path('dataset_split') / 'binary'  # beklenen çıktı klasörü
BATCH_SIZE = 32
IMG_SIZE = (128, 128)
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
# 1.5) Veri örneklerini göster
import matplotlib.pyplot as plt
import numpy as np

def show_examples(dataset, class_names, num=9):
    plt.figure(figsize=(8,8))
    for images, labels in dataset.take(1):
        images = images.numpy()
        labels = labels.numpy()
        n = min(num, images.shape[0])
        for i in range(n):
            ax = plt.subplot(3,3,i+1)
            img = images[i]
            img = (img * 255).astype('uint8')
            plt.imshow(img)
            plt.title(class_names[labels[i]])
            plt.axis('off')
    plt.show()

# Kullanım: eğitim setinden 9 örnek göster
show_examples(train_ds, class_names, num=9)

# 2) Veri yükleme: `image_dataset_from_directory` kullanarak train/val/test oluşturma
Bu yaklaşım klasör yapısını `class_name` olarak kullanır.

In [ ]:
train_dir = BASE_DIR / 'train'
val_dir = BASE_DIR / 'val'
test_dir = BASE_DIR / 'test'

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    labels='inferred',
    label_mode='int',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='int',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print('Classes:', class_names)

# 3) Ön işleme ve augmentation
- Normalize (0-1) ve opsiyonel data augmentation.

In [ ]:
normalization_layer = layers.Rescaling(1./255)  # [0,255] -> [0,1]
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def prepare(ds, training=False):
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(buffer_size=AUTOTUNE)

train_ds = prepare(train_ds, training=True)
val_ds = prepare(val_ds, training=False)
test_ds = prepare(test_ds, training=False)

# 4) Basit CNN modeli tasarımı
- Öğrenciler için anlaşılır ve küçük bir yapı.

In [ ]:
def make_model(input_shape=(*IMG_SIZE, 3)):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation='relu')(inputs)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(64, 3, activation='relu')(x)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(128, 3, activation='relu')(x)
    x = layers.MaxPool2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)  # binary
    model = keras.Model(inputs, outputs)
    return model

model = make_model()
model.summary()

# 5) Compile ve eğitim
- Binary crossentropy ve bazı callback'ler.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.ModelCheckpoint('best_binary_model.h5', save_best_only=True, monitor='val_loss'),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]

EPOCHS = 10
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

# 6) Kaydetme ve Yükleme
- `model.save()` ile kaydetme ve `keras.models.load_model()` ile geri yükleme.

In [ ]:
# Tam modeli kaydet
model.save('final_binary_model')
# Yükleme örneği:
# loaded = keras.models.load_model('final_binary_model')

# 7) Değerlendirme ve metrikler
- Test seti üzerinde tahmin, classification report ve confusion matrix.

In [ ]:
# Test üzerinde tahmin alma (tüm örnekleri tek bir array'a almak)
y_true = []
y_pred = []
for x_batch, y_batch in test_ds:
    preds = model.predict(x_batch)
    preds = (preds.ravel() > 0.5).astype(int)
    y_true.extend(y_batch.numpy().tolist())
    y_pred.extend(preds.tolist())

print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)
print('Confusion matrix:
', cm)

# Confusion matrix görselleştirme fonksiyonu
def plot_confusion_matrix(cm, classes):
    plt.figure(figsize=(5,5))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion matrix')
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment='center',
                 color='white' if cm[i, j] > thresh else 'black')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

plot_confusion_matrix(cm, class_names)

# 8) Grad-CAM (basit uygulama)
- Modelin son Conv katmanını kullanarak bir görüntü için ısı haritası çıkarma.

In [ ]:
# Grad-CAM yardımcı fonksiyonu
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# Örnek: test setinden bir batch alıp ilk görsel için Grad-CAM
for x_batch, y_batch in test_ds.take(1):
    img = x_batch[0:1]  # batch dim'li
    true_label = y_batch[0].numpy()
    break

# Modeldeki son conv katmanının adını bul (kendi modelinize göre ayarlayın)
for layer in model.layers[::-1]:
    if isinstance(layer, layers.Conv2D):
        last_conv_name = layer.name
        break
print('Last conv layer:', last_conv_name)

heatmap = make_gradcam_heatmap(img, model, last_conv_name)

# Isı haritasını orijinal görüntüye overlap etme
import cv2
def display_gradcam(img, heatmap, alpha=0.4):
    img = img.numpy().squeeze()
    img = (img * 255).astype('uint8')
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed = cv2.addWeighted(img, 1 - alpha, heatmap, alpha, 0)
    plt.figure(figsize=(6,6))
    plt.imshow(superimposed)
    plt.axis('off')

display_gradcam(img, heatmap)

# Son notlar
- Hücreleri sırayla çalıştırın.
- Eğer datasetiniz `dataset` içindeyse önce `split_dataset.py` scriptini çalıştırıp `dataset_split` oluşturun.
- `cv2` (opencv-python) yüklü değilse `pip install opencv-python` ile yükleyin.